# Sprint 1: MIND Data Preparation

Downloads MIND-small, builds FAISS index, clusters real users.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import yaml

with open('../configs/mind_small.yaml') as f:
    config = yaml.safe_load(f)
ds = config['dataset']
ir = config['ir']
print('Config loaded:', ds['name'])

In [ ]:
# Download MIND-small from https://msnews.github.io/
# Extract to data/MINDsmall_train/ and data/MINDsmall_dev/
# (manual step — the dataset requires agreeing to terms of use)
import os
assert os.path.exists(ds['train_news_path']), f"Missing: {ds['train_news_path']}"
assert os.path.exists(ds['train_behaviors_path']), f"Missing: {ds['train_behaviors_path']}"
print('MIND files present.')

In [ ]:
from data.mind_loader import load_news, load_behaviors, filter_active_users, temporal_split, cluster_users

news_df = load_news(ds['train_news_path'])
print(f'News articles: {len(news_df):,}')
print(news_df['coarse_category'].value_counts())

In [ ]:
behaviors_df = load_behaviors(ds['train_behaviors_path'], max_users=None)
print(f'Raw behavior rows: {len(behaviors_df):,}')
print(f'Unique users: {behaviors_df["user_id"].nunique():,}')

behaviors_df = filter_active_users(behaviors_df, min_clicks=ds['min_real_user_clicks'])
print(f'Active users (>={ds["min_real_user_clicks"]} clicks): {behaviors_df["user_id"].nunique():,}')

In [ ]:
train_df, test_df = temporal_split(behaviors_df, test_ratio=ds['test_ratio'])
print(f'Train rows: {len(train_df):,}  |  Test rows: {len(test_df):,}')

In [ ]:
cluster_df, cluster_dist = cluster_users(train_df, news_df, n_clusters=8)
print('Cluster distribution:')
print(cluster_df['archetype'].value_counts())
print('\nFractional distribution:')
for k, v in sorted(cluster_dist.items(), key=lambda x: -x[1]):
    print(f'  cluster {k}: {v:.2%}')

In [ ]:
from data.news_preprocessor import build_index

index, corpus_ids, embeddings = build_index(
    news_df,
    index_path=ds['index_path'],
    ids_path=ds['ids_path'],
    embeddings_path=ds['embeddings_path'],
    db_path=ds['db_path'],
    model_name=ir['model'],
    batch_size=ir['batch_size'],
)
print(f'Index built: {index.ntotal:,} vectors, embed_dim={embeddings.shape[1]}')

In [ ]:
# Sanity-check: search for a test query
import sqlite3
from sentence_transformers import SentenceTransformer
from data.news_preprocessor import search

model = SentenceTransformer(ir['model'])
db_con = sqlite3.connect(ds['db_path'], check_same_thread=False)

results = search('US election results 2019', model, index, corpus_ids, db_con, top_k=5)
for r in results:
    print(f"  [{r['coarse_category']}] {r['title'][:80]}  (score={r['score']:.3f})")

## Done
FAISS index and SQLite DB saved. Proceed to `02_simulation.ipynb`.